# URL-feature model sweeps

COSC 50523 — Deep Minds — Leiton Peterson

Converted from the team-provided week1/C1_sweeps_lr_rf_mlp.py. Uses the original experiment grids and log labels. Expected runs: 8 LR, 18 RF, and 12 MLP (38 total), evaluated on validation only.

Select a T4 GPU before running. Run cells in order. Each training cell appends results to the shared results_log.csv; avoid rerunning completed training cells, and coordinate with teammates so only one notebook writes the log at a time. The final check verifies the expected configuration coverage.

No experiment outputs are included yet.

## Cell 1 — setup (same in every notebook)

In [1]:
import sys
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/DeepMinds')   # shared folder: holds project_utils.py + torch_models.py
sys.path.append(str(DATA_DIR))
FIG_DIR = DATA_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
LOG = DATA_DIR / 'results_log.csv'                    # every run adds one row here

Mounted at /content/drive


## Cell 2 — load both feature sets

In [2]:
import itertools
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from project_utils import URL_FEATURES, load_master, get_feature_splits, evaluate_model, log_result
from torch_models import fit_mlp, predict_proba, count_params

df = load_master(DATA_DIR)
FEATURE_SETS = {
    'url22': URL_FEATURES,
    'url21': [f for f in URL_FEATURES if f != 'URLSimilarityIndex'],
}
DATA = {name: get_feature_splits(df, feats)[0] for name, feats in FEATURE_SETS.items()}

def tag(feature_set):
    return 'sweep' if feature_set == 'url22' else 'ablation: no URLSimilarityIndex'

## Cell 3 — Logistic Regression: C in {0.01, 0.1, 1, 10}   (about 1 minute)

In [3]:
for fs, C in itertools.product(DATA, [0.01, 0.1, 1.0, 10.0]):
    (Xtr, ytr), (Xva, yva) = DATA[fs]['train'], DATA[fs]['val']
    m = LogisticRegression(C=C, max_iter=1000, random_state=42).fit(Xtr, ytr)
    res = evaluate_model(f'LR C={C}', 'val', yva, m.predict_proba(Xva)[:, 1], show=False)
    log_result(res, {'family': 'LR', 'experiment': tag(fs), 'features': fs, 'split_col': 'split',
                     'C': C, 'penalty': 'l2', 'solver': 'lbfgs', 'max_iter': 1000}, LOG)
    print(fs, C, round(res['F1'], 4), 'missed:', res['False negatives'])

url22 0.01 0.9978 missed: 65
url22 0.1 0.9992 missed: 25
url22 1.0 0.9996 missed: 11
url22 10.0 0.9996 missed: 11
url21 0.01 0.9916 missed: 206
url21 0.1 0.9937 missed: 169
url21 1.0 0.995 missed: 143
url21 10.0 0.9958 missed: 123


## Cell 4 — Random Forest: n_estimators {50, 100, 300} x max_depth {10, 20, None}   (about 10–15 min)

In [4]:
for fs, n, depth in itertools.product(DATA, [50, 100, 300], [10, 20, None]):
    (Xtr, ytr), (Xva, yva) = DATA[fs]['train'], DATA[fs]['val']
    m = RandomForestClassifier(n_estimators=n, max_depth=depth, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    res = evaluate_model(f'RF n={n} depth={depth}', 'val', yva, m.predict_proba(Xva)[:, 1], show=False)
    log_result(res, {'family': 'RF', 'experiment': tag(fs), 'features': fs, 'split_col': 'split',
                     'n_estimators': n, 'max_depth': depth}, LOG)
    print(fs, n, depth, round(res['F1'], 4), 'missed:', res['False negatives'])

url22 50 10 0.9998 missed: 6
url22 50 20 0.9998 missed: 6


/content/drive/MyDrive/DeepMinds/project_utils.py:104: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new = pd.concat([pd.read_csv(log_file), new], ignore_index=True)


url22 50 None 0.9998 missed: 6
url22 100 10 0.9998 missed: 6
url22 100 20 0.9998 missed: 6


/content/drive/MyDrive/DeepMinds/project_utils.py:104: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new = pd.concat([pd.read_csv(log_file), new], ignore_index=True)


url22 100 None 0.9998 missed: 6
url22 300 10 0.9998 missed: 6
url22 300 20 0.9998 missed: 6


/content/drive/MyDrive/DeepMinds/project_utils.py:104: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new = pd.concat([pd.read_csv(log_file), new], ignore_index=True)


... (2 outputs left out)

url21 50 None 0.9958 missed: 85
url21 100 10 0.9951 missed: 124
url21 100 20 0.9965 missed: 98


/content/drive/MyDrive/DeepMinds/project_utils.py:104: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new = pd.concat([pd.read_csv(log_file), new], ignore_index=True)


url21 100 None 0.9959 missed: 83
url21 300 10 0.9951 missed: 125
url21 300 20 0.9965 missed: 98
url21 300 None 0.9959 missed: 86


/content/drive/MyDrive/DeepMinds/project_utils.py:104: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new = pd.concat([pd.read_csv(log_file), new], ignore_index=True)


## Cell 5 — MLP: hidden {[32], [64,32], [128,64,32]} x dropout {0, 0.3}

In [5]:
for fs, hidden, drop in itertools.product(DATA, [(32,), (64, 32), (128, 64, 32)], [0.0, 0.3]):
    model, hist, cfg = fit_mlp(DATA[fs], {'hidden': hidden, 'dropout': drop}, verbose=False)
    Xva, yva = DATA[fs]['val']
    res = evaluate_model(f'MLP {list(hidden)} drop={drop}', 'val', yva, predict_proba(model, Xva), show=False)
    log_result(res, {'family': 'MLP', 'experiment': tag(fs), 'features': fs, 'split_col': 'split',
                     'params': count_params(model), 'best_epoch': hist['best_epoch'], **cfg}, LOG)
    print(fs, hidden, drop, round(res['F1'], 4), 'missed:', res['False negatives'], 'epochs:', len(hist['loss']))

url22 (32,) 0.0 0.9997 missed: 9 epochs: 20
url22 (32,) 0.3 0.9996 missed: 11 epochs: 16
url22 (64, 32) 0.0 0.9997 missed: 9 epochs: 8
url22 (64, 32) 0.3 0.9997 missed: 9 epochs: 8
url22 (128, 64, 32) 0.0 0.9998 missed: 7 epochs: 11
url22 (128, 64, 32) 0.3 0.9997 missed: 8 epochs: 10
url21 (32,) 0.0 0.9967 missed: 99 epochs: 20
url21 (32,) 0.3 0.9965 missed: 102 epochs: 20
url21 (64, 32) 0.0 0.9971 missed: 89 epochs: 16
url21 (64, 32) 0.3 0.9972 missed: 86 epochs: 20
url21 (128, 64, 32) 0.0 0.9978 missed: 67 epochs: 16
url21 (128, 64, 32) 0.3 0.9966 missed: 100 epochs: 7


## Cell 6 — summary: best setting per model (by validation F1)

In [6]:
log = pd.read_csv(LOG)
best = (log[log['Split'] == 'val'].sort_values('F1', ascending=False)
        .groupby(['hp_family', 'hp_features']).head(1))
print(best[['hp_family', 'hp_features', 'Model', 'F1', 'Recall', 'False negatives', 'False positives']]
      .round(4).to_string(index=False))

hp_family hp_features                      Model     F1  Recall  False negatives  False positives
       RF       url22              Random Forest 0.9998  0.9996                6                0
      MLP       url22 MLP [128, 64, 32] drop=0.0 0.9998  0.9995                7                0
       LR       url22                  LR C=10.0 0.9996  0.9993               11                0
      CNN     raw URL      Char-CNN embed_dim=64 0.9979  0.9959               62                2
      CNN     raw_url                   Char-CNN 0.9978  0.9956               66                1
      MLP       url21 MLP [128, 64, 32] drop=0.0 0.9978  0.9956               67                0
       RF       url21          RF n=100 depth=20 0.9965  0.9935               98                8
       LR       url21                  LR C=10.0 0.9958  0.9919              123                3


## Verify the shared log contains all 38 configurations

In [7]:
# Check every expected model/feature-set combination in the saved log.
log = pd.read_csv(LOG)
expected = []
for fs in ['url22', 'url21']:
    expected += [('LR', fs, f'LR C={C}') for C in [0.01, 0.1, 1.0, 10.0]]
    expected += [('RF', fs, f'RF n={n} depth={depth}')
                 for n in [50, 100, 300] for depth in [10, 20, None]]
    expected += [('MLP', fs, f'MLP {list(hidden)} drop={drop}')
                 for hidden in [(32,), (64, 32), (128, 64, 32)]
                 for drop in [0.0, 0.3]]

valid = log[
    (log['Split'] == 'val') &
    (((log['hp_features'] == 'url22') & (log['hp_experiment'] == 'sweep')) |
     ((log['hp_features'] == 'url21') &
      (log['hp_experiment'] == 'ablation: no URLSimilarityIndex')))
].copy()
keys = ['hp_family', 'hp_features', 'Model']
expected_index = pd.MultiIndex.from_tuples(expected, names=keys)
actual_index = pd.MultiIndex.from_frame(valid[keys])
matched = valid[actual_index.isin(expected_index)]
missing = expected_index.difference(pd.MultiIndex.from_frame(matched[keys]))
print('Expected configurations:', len(expected))
print('Completed unique configurations:', len(matched.drop_duplicates(keys)))
print('Duplicate log rows:', int(matched.duplicated(keys).sum()))
if len(missing):
    print('Still missing:', list(missing))
assert len(missing) == 0, 'Some planned experiments are missing from the shared log.'
assert not matched.duplicated(keys).any(), 'Duplicate runs need review before plotting; do not delete other team results.'
assert matched[['Accuracy', 'Precision', 'Recall', 'F1', 'False positives', 'False negatives']].notna().all().all()
display(matched.groupby(['hp_family', 'hp_features']).size().unstack())
print('All 38 configurations are logged. Review the summary and add your observations.')

Expected configurations: 38
Completed unique configurations: 38
Duplicate log rows: 0


hp_features,url21,url22
hp_family,,
LR,4,4
MLP,6,6
RF,9,9


All 38 configurations are logged. Review the summary and add your observations.


## Findings

Completed all 38 planned validation experiments: 8 logistic-regression,
18 Random Forest, and 12 MLP runs across the url22 and url21 feature sets.
The shared results log contains every expected configuration with no
duplicates among these runs.

Removing URLSimilarityIndex reduced performance across the tested model
families. Among the tested MLP configurations, hidden layers [128, 64, 32]
with dropout 0 achieved the highest displayed F1 for both feature sets,
missing 7 phishing examples with url22 and 67 with url21.

Configurations are summarized by validation F1, with recall and false
positives also reported. These results use one random seed and do not
establish that URLSimilarityIndex contains leakage.

No test-set evaluation was performed.